In [1]:
import os
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"

## (1) Load Model & Weights from HuggingFace

In [2]:
import json
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from model import Mamba, ModelArgs

def load_flax_mamba(model_name: str):
    # 1. 下载并解析 config.json 动态获取参数
    config_path = hf_hub_download(repo_id=model_name, filename="config.json")
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    args = ModelArgs(
        d_model=config.get('hidden_size', config.get('d_model', 768)),
        n_layer=config.get('num_hidden_layers', config.get('n_layer', 24)),
        vocab_size=config.get('vocab_size', 50277)
    )
    model = Mamba(args)
    
    # 2. 下载 safetensors 权重并转换为 Flax 参数字典
    model_path = hf_hub_download(repo_id=model_name, filename="model.safetensors")
    params = {}
    
    with safe_open(model_path, framework="np", device="cpu") as f:
        # 提取 Embedding (注意 s) 和最后的 Norm
        params['embedding'] = {'embedding': jnp.array(f.get_tensor('backbone.embeddings.weight'))}
        params['norm_f'] = {'weight': jnp.array(f.get_tensor('backbone.norm_f.weight'))}
        
        # 遍历每一层进行权重映射
        for i in range(args.n_layer):
            layer_name = f'layers_{i}'
            pt_prefix = f'backbone.layers.{i}'
            
            # 卷积层权重维度转换: PyTorch(C_out, C_in, L) -> Flax(L, C_in, C_out)
            conv_weight = f.get_tensor(f'{pt_prefix}.mixer.conv1d.weight')
            conv_weight = jnp.transpose(conv_weight, (2, 1, 0))
            
            # 线性层需要转置 (.T)
            params[layer_name] = {
                'norm': {'weight': jnp.array(f.get_tensor(f'{pt_prefix}.norm.weight'))},
                'mixer': {
                    'A_log': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.A_log')),
                    'D': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.D')),
                    'in_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.in_proj.weight')).T},
                    'conv1d': {
                        'kernel': conv_weight,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.conv1d.bias'))
                    },
                    'x_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.x_proj.weight')).T},
                    'dt_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.weight')).T,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.bias'))
                    },
                    'out_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.out_proj.weight')).T}
                }
            }
            
    return model, params

# 加载模型和分词器
pretrained_model_name = 'state-spaces/mamba-130m-hf'
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')
model, params = load_flax_mamba(pretrained_model_name)
print("预训练模型参数与配置已成功加载并转换！")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


预训练模型参数与配置已成功加载并转换！


## (2) Generate Text

In [3]:
from functools import partial

@partial(jax.jit, static_argnames=('top_k',))
def forward_step(params, input_ids, top_k=40):
    logits = model.apply({'params': params}, input_ids)
    next_token_logits = logits[:, -1, :]
    return next_token_logits

def generate(params, tokenizer, prompt: str, n_tokens_to_gen: int = 50, sample: bool = True, top_k: int = 40, seed: int = 10086):
    key = jax.random.PRNGKey(seed)
    input_ids = jnp.array(tokenizer(prompt, return_tensors='np').input_ids)
    
    for _ in range(n_tokens_to_gen):
        next_token_logits = forward_step(params, input_ids, top_k)
        
        if top_k is not None:
            values, _ = jax.lax.top_k(next_token_logits, k=top_k)
            kth_values = values[:, -1:]
            next_token_logits = jnp.where(next_token_logits < kth_values, -1e9, next_token_logits)
            
        probs = jax.nn.softmax(next_token_logits, axis=-1)
        
        if sample:
            key, subkey = jax.random.split(key)
            next_indices = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)[:, None]
        else:
            next_indices = jnp.argmax(probs, axis=-1)[:, None]
            
        input_ids = jnp.concatenate([input_ids, next_indices], axis=1)

    return tokenizer.decode(input_ids[0].tolist())


In [4]:
print(generate(params, tokenizer, 'Mamba is the'))

Mamba is the smallest of the seven species, with a length typically 1.5 meters to 2 meters (about 5 feet) in adult female and 1 to 2.5 meters (about 8 feet) in adult male.

Habitat

The species


In [5]:
print(generate(params, tokenizer, 'The meaning of life is '))

The meaning of life is 
"life as a product of human nature".

When humans are happy, each of them has been given a momentary liberty to take a few moments or a certain number of minutes each day to live in their lives.

"Life


In [6]:
print(generate(params, tokenizer, 'def reverse_string('))

def reverse_string(self, data_to_reverse):
	"""
	Retrieve a reverse string as a list of strings

	@return The reverse list
	"""
	self.data_to_reverse = self.get_array(self


In [7]:
print(generate(params, tokenizer, 'The meaning of life is ', seed=114514))

The meaning of life is 
"a state of being and the state of having a state of being"
(Vergniaud, 2003)

In an interview, Jean-Luc Godard recalled the meaning of life as an "entirely new thing,
